In [ ]:
!pip install streamlit pillow matplotlib
!pip install streamlit
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.8 MB/s eta 0:00:00


In [ ]:
app_code = """
import torch
import torch.optim as optim
from torch import nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import streamlit as st

# Image Preprocessing Function
def load_image(image, max_size=400):
    image = Image.open(image).convert("RGB")
    w, h = image.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        new_w = int(w * scale)
        new_h = int(h * scale)
        image = image.resize((new_w, new_h))

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image

# Get Pretrained VGG19 Model
def get_model():
    vgg = models.vgg19(pretrained=True).features
    for param in vgg.parameters():
        param.requires_grad = False  # Freeze parameters for inference
    return vgg

# Gram Matrix Function
def gram_matrix(input):
    batch_size, channels, height, width = input.size()
    features = input.view(batch_size * channels, height * width)
    gram = torch.mm(features, features.t())
    return gram.div(channels * height * width)

class LossFunctions:
    def __init__(self, style_weight=1000000, content_weight=1):
        self.style_weight = style_weight
        self.content_weight = content_weight

    def content_loss(self, content, target):
        return torch.nn.functional.mse_loss(content, target)

    def style_loss(self, style, target):
        gram_style = gram_matrix(style)
        gram_target = gram_matrix(target)
        return torch.nn.functional.mse_loss(gram_target, gram_style)

# Style Transfer Function
def run_style_transfer(content_image, style_image, model, num_steps=500, style_weight=1000000, content_weight=1):
    target_image = content_image.clone().requires_grad_(True)
    optimizer = optim.LBFGS([target_image])  # Using L-BFGS optimizer

    loss_functions = LossFunctions(style_weight, content_weight)

    content_layers = ['21']
    style_layers = ['0', '5', '10', '19']

    model = model.eval()

    def closure():
        optimizer.zero_grad()
        target_image.data.clamp_(0, 1)

        content_loss = 0
        style_loss = 0

        x = content_image
        target = target_image

        for i, layer in enumerate(model):
            x = layer(x)
            target = layer(target)

            if str(i) in content_layers:
                content_loss += loss_functions.content_loss(x, target)

            if str(i) in style_layers:
                style_loss += loss_functions.style_loss(x, target)

        total_loss = content_loss + style_loss
        total_loss.backward()
        return total_loss

    for step in range(num_steps):
        optimizer.step(closure)

    return target_image

# Streamlit UI code (without wrapping it in a function)
st.title("Neural Style Transfer")

content_image = st.file_uploader("Upload Content Image", type=['png', 'jpg', 'jpeg'])
style_image = st.file_uploader("Upload Style Image", type=['png', 'jpg', 'jpeg'])

if content_image and style_image:
    content_image = load_image(content_image)
    style_image = load_image(style_image)

    model = get_model()
    stylized_image = run_style_transfer(content_image, style_image, model)

    st.image(content_image.squeeze(0).permute(1, 2, 0).numpy(), caption="Original Content Image", use_column_width=True)
    st.image(style_image.squeeze(0).permute(1, 2, 0).numpy(), caption="Style Image", use_column_width=True)
    st.image(stylized_image.squeeze(0).permute(1, 2, 0).detach().numpy(), caption="Stylized Image", use_column_width=True)
"""

# Save the code to app.py
with open("app.py", "w") as f:
    f.write(app_code)

In [ ]:
!streamlit run app.py --server.port 8501 &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.150.255.143:8501



In [ ]:
from pyngrok import ngrok

# Create a public URL for the Streamlit app running on port 8501
public_url = ngrok.connect(8501)
public_url